# B. Traditional Text Classification

In [1]:
# Fetch dataset
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset", output_dir="data")

print("Path to dataset files:", path)

/home/master/.pyenv/versions/3.10.8/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: data


Exploring the DS

In [2]:
import pandas as pd

In [3]:
# Load into pandas
train_df = pd.read_csv(f"{path}/train.csv")
test_df = pd.read_csv(f"{path}/test.csv")

In [4]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Class Index  120000 non-null  int64 
 1   Title        120000 non-null  object
 2   Description  120000 non-null  object
dtypes: int64(1), object(2)
memory usage: 2.7+ MB


In [5]:
train_df.head(3)

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...


In [6]:
# Text classes
train_df["Class Index"].unique()
# 1-World, 2-Sports, 3-Business, 4-Sci/Tech

array([3, 4, 2, 1])

In [7]:
# CLass text split
train_df["Class Index"].value_counts()

Class Index
3    30000
4    30000
2    30000
1    30000
Name: count, dtype: int64

In [8]:
# Build Train and Test dataframes
X_train_df, y_train_df = train_df["Description"], train_df["Class Index"]
X_test_df, y_test_df = test_df["Description"], test_df["Class Index"]

In [9]:
# Sanity check
X_train_df.shape, y_train_df.shape, X_test_df.shape, y_test_df.shape

((120000,), (120000,), (7600,), (7600,))

---

### Train Multinomial NB using Tfidf Matrix - **word 1-grams**

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
# Check accuracy
from sklearn.metrics import accuracy_score
# Measure time
import time

In [11]:
# Build TIDF Matrix vectorizer
start_time = time.time()
vectorizer = TfidfVectorizer(ngram_range=(1,1), lowercase=True, analyzer="word")
X_train = vectorizer.fit_transform(X_train_df)
X_test = vectorizer.transform(X_test_df)
end_time = time.time()
print(f"Vectorization time: {end_time - start_time:.4} sec.")

Vectorization time: 1.75 sec.


In [12]:
X_train.shape, X_test.shape

((120000, 60734), (7600, 60734))

This is the TFIDF matrix: 120K documents, 60734 terms.

In [13]:
# Dictionary size 60734
list(vectorizer.vocabulary_)[0:10]

['reuters',
 'short',
 'sellers',
 'wall',
 'street',
 'dwindling',
 'band',
 'of',
 'ultra',
 'cynics']

In [16]:
# Train
start_time = time.time()
clf = MultinomialNB()
clf.fit(X_train, y_train_df)
end_time = time.time()
print(f"NB train time: {end_time - start_time:.4} sec.")

NB train time: 0.02201 sec.


In [17]:
start_time = time.time()
preds = clf.predict(X_test)
end_time = time.time()
print(f"NB Inference time: {end_time - start_time:.4} sec.")

NB Inference time: 0.003077 sec.


In [18]:
# Missclassified text
print(f"Actual category: {y_test_df[3]}")
print(f"Predicted cat: {preds[3]}")

Actual category: 4
Predicted cat: 2


In [20]:
# Accuracy
accuracy_score(y_test_df, preds)

0.8935526315789474

In [21]:
# Dimensionality
len(vectorizer.vocabulary_)

60734

In [22]:
# Test on custom input
custom_news = [
    "The game was cancelled due to heavy rain.",
    "The singularity is nearer but nobody knows when or how it will happen.",
    "Low and slow is the optimal strategy for passive investors.",
    "Visit afghanistan for your next, memorable, vacation."
]

X_custom = vectorizer.transform(custom_news)
print(clf.predict(X_custom))

[2 4 3 1]


In [43]:
# Get missclassified indexes
diffs = (~np.equal(preds, y_test_df)).astype(int)
missed_indices_nb_1word = np.flatnonzero(diffs)
len(missed_indices_nb_1word)

756

### Train Multinomial NB using Tfidf Matrix - **word 3-grams**

In [25]:
# Build TIDF Matrix vectorizer
start_time = time.time()
vectorizer = TfidfVectorizer(ngram_range=(3, 3), lowercase=True, analyzer="char_wb")
X_train = vectorizer.fit_transform(X_train_df)
X_test = vectorizer.transform(X_test_df)
end_time = time.time()
print(f"Vectorization time: {end_time - start_time:.4} sec.")
print(f"Train DF shapes: {X_train.shape}, {X_test.shape}")
print(f"Dimensionality/Dictionary: {len(vectorizer.vocabulary_)}")

# Training
start_time = time.time()
clf = MultinomialNB()
clf.fit(X_train, y_train_df)
end_time = time.time()
print(f"Training time: {end_time - start_time:.4} sec.")

# Inference
start_time = time.time()
preds = clf.predict(X_test)
end_time = time.time()
print(f"Inference time: {end_time - start_time:.4} sec.")

# Accuracy
print(f"Accuracy: {accuracy_score(y_test_df, preds)}")

# Missclassified text
print(f"Actual category: {y_test_df[3]}")
print(f"Predicted cat: {preds[3]}")

Vectorization time: 7.215 sec.
Train DF shapes: (120000, 27301), (7600, 27301)
Dimensionality/Dictionary: 27301
Training time: 0.0506 sec.
Inference time: 0.003696 sec.
Accuracy: 0.8559210526315789
Actual category: 4
Predicted cat: 2


In [44]:
# Get missclassified indexes
diffs = (~np.equal(preds, y_test_df)).astype(int)
missed_indices_nb_3word = np.flatnonzero(diffs)
len(missed_indices_nb_3word)

756

---

### Train Linear SVC using Tfidf Matrix - **word 1-grams**

In [26]:
from sklearn.svm import LinearSVC

In [27]:
# Vectorization
vectorizer = TfidfVectorizer(ngram_range=(1, 1), lowercase=True, analyzer="word")
X_train = vectorizer.fit_transform(X_train_df)
X_test = vectorizer.transform(X_test_df)
print(f"Vectorization time: {end_time - start_time:.4} sec.")
print(f"Train DF shapes: {X_train.shape}, {X_test.shape}")
print(f"Dimensionality/Dictionary: {len(vectorizer.vocabulary_)}")

# Training
start_time = time.time()
clf = LinearSVC()
clf.fit(X_train, y_train_df)
end_time = time.time()
print(f"Training time: {end_time - start_time:.4} sec.")

start_time = time.time()
preds = clf.predict(X_test)
end_time = time.time()
print(f"Inference time: {end_time - start_time:.4} sec.")

# Accuracy
print(f"Accuracy: {accuracy_score(y_test_df, preds)}")

# Missclassified text
print(f"Actual category: {y_test_df[3]}")
print(f"Predicted cat: {preds[3]}")

Vectorization time: 0.003696 sec.
Train DF shapes: (120000, 60734), (7600, 60734)
Dimensionality/Dictionary: 60734
Training time: 4.358 sec.
Inference time: 0.001762 sec.
Accuracy: 0.9102631578947369
Actual category: 4
Predicted cat: 1


In [45]:
# Get missclassified indexes
diffs = (~np.equal(preds, y_test_df)).astype(int)
missed_indices_svc_1word = np.flatnonzero(diffs)
len(missed_indices_svc_1word)

756

### Train Linear SVC using Tfidf Matrix - **word 3-grams**

In [29]:
# Vectorization
vectorizer = TfidfVectorizer(ngram_range=(3, 3), lowercase=True, analyzer="char_wb")
X_train = vectorizer.fit_transform(X_train_df)
X_test = vectorizer.transform(X_test_df)
print(f"Vectorization time: {end_time - start_time:.4} sec.")
print(f"Train DF shapes: {X_train.shape}, {X_test.shape}")
print(f"Dimensionality/Dictionary: {len(vectorizer.vocabulary_)}")

# Training
start_time = time.time()
clf = LinearSVC()
clf.fit(X_train, y_train_df)
end_time = time.time()
print(f"Training time: {end_time - start_time:.4} sec.")

start_time = time.time()
preds = clf.predict(X_test)
end_time = time.time()
print(f"Inference time: {end_time - start_time:.4} sec.")

# Accuracy
print(f"Accuracy: {accuracy_score(y_test_df, preds)}")

# Missclassified text
print(f"Actual category: {y_test_df[3]}")
print(f"Predicted cat: {preds[3]}")

Vectorization time: 0.004436 sec.
Train DF shapes: (120000, 27301), (7600, 27301)
Dimensionality/Dictionary: 27301
Training time: 11.45 sec.
Inference time: 0.004447 sec.
Accuracy: 0.9005263157894737
Actual category: 4
Predicted cat: 2


In [46]:
# Get missclassified indexes
diffs = (~np.equal(preds, y_test_df)).astype(int)
missed_indices_svc_3word = np.flatnonzero(diffs)
len(missed_indices_svc_3word)

756

---

Find common missclassified texts.

In [30]:
import numpy as np

In [32]:
diffs = (~np.equal(preds, y_test_df)).astype(int)
missed_indices = np.flatnonzero(diffs)
missed_indices

# Example of COMMON missclassified text - All models
print(X_test_df[3])
# Expected
print(f"Actual category: {y_test_df[3]}")
print(f"Predicted cat: {preds[3]}")

AP - It's barely dawn when Mike Fitzpatrick starts his shift with a blur of colorful maps, figures and endless charts, but already he knows what the day will bring. Lightning will strike in places he expects. Winds will pick up, moist places will dry and flames will roar.
Actual category: 4
Predicted cat: 2


In [59]:
# Common missclassified indexes - test ds
common_missed_indices = (missed_indices_nb_1word & missed_indices_nb_3word & missed_indices_svc_1word & missed_indices_svc_3word)
len(common_missed_indices)

756

In [60]:
# Accumulate missed ctaegories - test ds
all_missed_categories = [ y_test_df[i] for i in common_missed_indices ]

In [63]:
# How many missed per category
for i in set(all_missed_categories):
    print(f"{all_missed_categories.count(i)} missed in category: {i}")

192 missed in category: 1
64 missed in category: 2
260 missed in category: 3
240 missed in category: 4
